In [1]:
import os
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
directory = '../src'

In [3]:
def analyze_code(file_path):
    with open(file_path, 'r') as file:
        lines = file.readlines()

    total_lines = len(lines)
    comment_lines = sum(1 for line in lines if line.strip().startswith('#'))
    docstring_lines = sum(1 for line in lines if line.strip(
    ).startswith('"""') or line.strip().startswith("'''"))
    empty_lines = sum(1 for line in lines if not line.strip())

    code_lines = total_lines - comment_lines - docstring_lines - empty_lines

    return {
        'total_lines': total_lines,
        'comment_lines': comment_lines,
        'docstring_lines': docstring_lines,
        'empty_lines': empty_lines,
        'code_lines': code_lines,
        'comment_percentage': (comment_lines / total_lines) * 100 if total_lines > 0 else 0,
        'docstring_percentage': (docstring_lines / total_lines) * 100 if total_lines > 0 else 0,
        'code_percentage': (code_lines / total_lines) * 100 if total_lines > 0 else 0
    }


def analyze_directory(directory):
    metrics = []
    for root, _, files in os.walk(directory):
        for file in files:
            if file.endswith('.py'):
                file_path = os.path.join(root, file)
                file_metrics = analyze_code(file_path)
                file_metrics['file_path'] = file_path
                file_metrics['folder_path'] = root
                metrics.append(file_metrics)
    return metrics


def aggregate_metrics_by_folder(df):
    folder_metrics = df.groupby('folder_path').sum()
    folder_metrics['comment_percentage'] = (
        folder_metrics['comment_lines'] / folder_metrics['total_lines']) * 100
    folder_metrics['docstring_percentage'] = (
        folder_metrics['docstring_lines'] / folder_metrics['total_lines']) * 100
    folder_metrics['code_percentage'] = (
        folder_metrics['code_lines'] / folder_metrics['total_lines']) * 100
    folder_metrics.reset_index(inplace=True)
    return folder_metrics

In [4]:
metrics = analyze_directory(directory)

# Create a DataFrame from the metrics
df = pd.DataFrame(metrics)

# # Save DataFrame to a CSV file (optional)
# df.to_csv('code_metrics.csv', index=False)

# Aggregate metrics by folder
folder_metrics = aggregate_metrics_by_folder(df)

# # Save folder-level DataFrame to a CSV file (optional)
# folder_metrics.to_csv('folder_metrics.csv', index=False)

In [23]:
df.describe()

,total_lines,comment_lines,docstring_lines,empty_lines,code_lines,comment_percentage,docstring_percentage,code_percentage
count,39.000000,39.000000,39.000000,39.000000,39.000000,39.000000,39.000000,39.000000
mean,205.641026,19.974359,11.102564,37.923077,136.641026,6.804610,4.270153,69.166290
std,246.798713,29.071135,15.520436,44.114913,163.447640,6.090109,2.438151,10.342440
min,5.000000,0.000000,0.000000,0.000000,3.000000,0.000000,0.000000,52.668213
25%,44.000000,0.000000,2.000000,8.000000,31.000000,0.000000,2.658451,62.460317
50%,92.000000,7.000000,4.000000,18.000000,61.000000,6.493506,4.237288,68.539326
75%,232.500000,24.000000,12.000000,47.000000,157.000000,11.938296,6.186224,73.432602
max,915.000000,117.000000,51.000000,164.000000,583.000000,18.691589,9.424084,100.000000


In [24]:
folder_metrics.describe()

,total_lines,comment_lines,docstring_lines,empty_lines,code_lines,comment_percentage,docstring_percentage,code_percentage
count,4.000000,4.000000,4.00000,4.000000,4.000000,4.000000,4.000000,4.000000
mean,2005.000000,194.750000,108.25000,369.750000,1332.250000,8.131301,4.838567,68.369108
std,1794.165173,176.021542,98.64203,312.501067,1216.492328,4.327539,1.334929,6.413703
min,244.000000,6.000000,7.00000,43.000000,188.000000,2.459016,2.868852,61.666667
25%,952.750000,74.250000,49.00000,188.500000,641.000000,6.733341,4.691141,65.374615
50%,1674.500000,188.500000,94.50000,329.500000,1062.000000,8.551613,5.326041,67.380293
75%,2726.750000,309.000000,153.75000,510.750000,1753.250000,9.949573,5.473468,70.374787
max,4427.000000,396.000000,237.00000,777.000000,3017.000000,12.962963,5.833333,77.049180
